# Imports

In [ ]:
import torch
import re
from tqdm import tqdm
import os
import collections
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, PeftModel
!pip install accelerate datasets --quiet
!pip install trl --quiet 
from trl import SFTTrainer, SFTConfig, GRPOTrainer, GRPOConfig
from google.colab import drive
!pip install --upgrade torchao


drive.mount('/content/drive')
drive_save_path_sft = "/content/drive/MyDrive/qwen-gsm8k-sft-final"
drive_save_path_grpo = "/content/drive/MyDrive/qwen-gsm8k-grpo-final"

# Model Load

In [ ]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="auto"
)

# Make sure we don't load multiple times or intiialize LoRa after loading weights
loaded = 0

In [ ]:
if os.path.exists(drive_save_path_sft):
    print(f"Loading saved LoRA adapter from {drive_save_path_sft}...")
    model = PeftModel.from_pretrained(model, drive_save_path_sft, is_trainable=True)
    print("Successfully loaded saved model")
    loaded = 1
else:
    print(f"Warning: No saved model found at {drive_save_path_sft}.")

In [ ]:
if os.path.exists(drive_save_path_grpo):
    if (loaded == 1):
      print(f"Already loaded SFT.")
    else:
        print(f"Loading saved LoRA adapter from {drive_save_path_grpo}...")
        model = PeftModel.from_pretrained(model, drive_save_path_grpo, is_trainable=True)
        print("Successfully loaded saved model")
        loaded = 1
else:
    print(f"Warning: No saved model found at {drive_save_path_grpo}.")

# LoRa Setup

In [ ]:
if (loaded == 0):
  lora_config = LoraConfig(
      r=32,
      lora_alpha=64,
      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
      lora_dropout=0.05,
      bias="none",
      task_type="CAUSAL_LM"
  )
  model = get_peft_model(model, lora_config)
  model.print_trainable_parameters()
else:
    print("skipping LoRA adapter initialization since a saved model was loaded.")

# Dataset Import

In [ ]:
# Load a small subset of the GSM8k dataset for training
train_dataset = load_dataset("openai/gsm8k", "main", split="train[:3000]")

test_dataset = load_dataset("openai/gsm8k", "main", split="test")


In [ ]:
train_dataset_sft  = train_dataset.select(range(1000))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_chat_template(example):
    # Format the SFT train answers to include <think> tags
    parts = example["answer"].split("####")
    reasoning = parts[0].strip()
    final_answer = parts[1].strip() if len(parts) > 1 else ""

    formatted_answer = f"<think>\n{reasoning}\n</think>\n#### {final_answer}"

    messages = [
        {"role": "system", "content": "You are a smart math assistant. Solve the math problem. Put reasoning inside <think> and </think>, then give the final answer after ####."},
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": formatted_answer}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

formatted_train_dataset_sft = train_dataset_sft.map(format_chat_template)

In [ ]:
train_dataset_sft  = train_dataset.select(range(1000))

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_chat_template(example):
    messages = [
        {"role": "system", "content": "You are a smart math assistant. Solve the problem step-by-step."},
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

formatted_train_dataset_sft = train_dataset_sft.map(format_chat_template)

In [ ]:
train_dataset_grpo = train_dataset.select(range(1000, 2000)) # only use 1000 to save compute units
def format_grpo_prompt(example):
    messages = [
        {"role": "system", "content": "You are smart math assitant. Solve the math problem. Put reasoning inside <think> and </think>, then give the final answer after ####."},
        {"role": "user", "content": example["question"]}
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    parts = example["answer"].split("####")
    reasoning = parts[0].strip()
    final_answer = parts[1].strip() if len(parts) > 1 else ""

    formatted_answer = f"<think>\n{reasoning}\n</think>\n#### {final_answer}"
    
    return {
        "prompt": prompt,
        "answer": formatted_answer
    }

formatted_train_dataset_grpo = train_dataset_grpo.map(format_grpo_prompt)
print(f"Prepared {len(formatted_train_dataset_grpo)} examples for GRPO training.")

In [ ]:
train_dataset_grpo = train_dataset.select(range(1000, 2000)) # only use 1000 to save compute units
def format_grpo_prompt(example):
    messages = [
        {"role": "system", "content": "You are smart math assitant. Solve the math problem. Put reasoning inside <think> and </think>, then give the final answer after ####."},
        {"role": "user", "content": example["question"]}
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return {
        "prompt": prompt,
        "answer": example["answer"]
    }

formatted_train_dataset_grpo = train_dataset_grpo.map(format_grpo_prompt)
print(f"Prepared {len(formatted_train_dataset_grpo)} examples for GRPO training.")

# SFT Setup

In [ ]:
training_args = SFTConfig(
    output_dir="./qwen-gsm8k-sft",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    logging_steps=10,
    num_train_epochs=1,
    fp16=True,
    dataset_text_field="text",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_train_dataset_sft,
    args=training_args,
    processing_class=tokenizer,
)

# GRPO Setup

In [ ]:
def extract_gsm8k_answer(text):
    if "####" in text:
        return text.split("####")[-1].strip()
    numbers = re.findall(r"-?\d+\.?\d*", text.replace(",", ""))
    return numbers[-1] if numbers else ""

def correctness_reward(completions, answer, **kwargs):
    rewards = []
    for completion, gold in zip(completions, answer):
        pred_ans = extract_gsm8k_answer(completion)
        gold_ans = extract_gsm8k_answer(gold)
        rewards.append(1.0 if pred_ans == gold_ans else 0.0)
    return rewards

def format_reward(completions, **kwargs):
    rewards = []
    for completion in completions:
        has_think = "<think>" in completion and "</think>" in completion
        has_final = "####" in completion
        rewards.append(0.1 if has_think and has_final else 0.0)
    return rewards

In [ ]:
grpo_training_args = GRPOConfig(
    output_dir="./qwen-gsm8k-grpo",
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=8,
    generation_batch_size=8,
    beta=0.04,
    logging_steps=10,
    save_steps=100,
    num_train_epochs=1,
    fp16=True,
    temperature=0.8,
    max_completion_length=512,
)

grpo_trainer = GRPOTrainer(
    model=model,
    args=grpo_training_args,
    train_dataset=formatted_train_dataset_grpo,
    reward_funcs=[correctness_reward, format_reward]
)

# Eval Functions

In [ ]:
def test_model_inference(current_model, current_tokenizer, question,
                         system_content="You are a smart math assistant. Solve the problem step-by-step."):
    current_model.eval()
    messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": question}
    ]

    text = current_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = current_tokenizer([text], return_tensors="pt").to(current_model.device)

    with torch.no_grad():
        outputs = current_model.generate(**inputs, max_new_tokens=512, temperature=0.8, do_sample=True)

    generated_ids = outputs[0][len(inputs.input_ids[0]):]
    response = current_tokenizer.decode(generated_ids, skip_special_tokens=True)

    current_model.train()
    return response

sample_question = "If I have 12 lemons and eat 6, then buy 5 more, how many lemons do I have?"

In [ ]:
def extract_think_block(text):
    match = re.search(r"<think>(.*?)</think>", text, re.DOTALL)
    return match.group(1).strip() if match else ""

def extract_ground_truth(answer_str):
    return answer_str.split("####")[-1].strip()

def extract_prediction(prediction_str):
    if "####" in prediction_str:
        return prediction_str.split("####")[-1].strip().replace(',', '')
    numbers = re.findall(r'-?\d+(?:\.\d+)?', prediction_str.replace(',', ''))
    if numbers:
        return numbers[-1]
    return None

BACKTRACK_PHRASES = [
    r"\bwait\b", r"\boops\b", r"let me recalculate", r"let me re-?check",
    r"actually,", r"that doesn'?t (seem|make sense)", r"i made an error",
    r"alternatively,", r"hmm,", r"let me try again", r"correction:",
    r"i was wrong", r"reconsider", r"on second thought",
]
BACKTRACK_PATTERN = re.compile("|".join(BACKTRACK_PHRASES), re.IGNORECASE)

# Eval prompts
SYSTEM_BASE  = "You are a smart math assistant. Solve the problem step-by-step and give the final answer after ####."
SYSTEM_THINK = "You are a smart math assistant. Solve the math problem. Put reasoning inside <think> and </think>, then give the final answer after ####."


In [ ]:
def _pass_at_k_batch(current_model, current_tokenizer, inputs, k):
    input_ids_k = inputs["input_ids"].repeat(k, 1)
    attn_k = inputs["attention_mask"].repeat(k, 1)
    try:
        with torch.no_grad():
            out_k = current_model.generate(
                input_ids=input_ids_k, attention_mask=attn_k,
                max_new_tokens=512, temperature=0.8, do_sample=True
            )
    except torch.cuda.OutOfMemoryError: # On OOM, fallback to sequential generation to at least get some results. Not a problem on Colab
        torch.cuda.empty_cache()
        out_k = []
        for _ in range(k):
            with torch.no_grad():
                out_k.append(current_model.generate(
                    **inputs, max_new_tokens=512, temperature=0.8, do_sample=True
                )[0])

    prompt_len = inputs["input_ids"].shape[1]
    answers = []
    for i in range(k):
        seq = out_k[i]
        gen_ids = seq[prompt_len:]
        response = current_tokenizer.decode(gen_ids, skip_special_tokens=True)
        answers.append(extract_prediction(response))
    return answers

In [ ]:
def analyze_standard(current_model, current_tokenizer, dataset, label, k=8, system_prompt=SYSTEM_BASE):
    print(f"\n{'#'*60}\n  STANDARD ANALYSIS: {label}\n{'#'*60}")
    current_model.eval()

    total = accuracy_correct = pass_at_4_hits = pass_at_8_hits = majority_hits = 0

    for example in tqdm(dataset, desc=f"Evaluating ({label})"):
        question = example["question"]
        gt_answer = extract_ground_truth(example["answer"])

        messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": question}]
        text = current_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = current_tokenizer([text], return_tensors="pt").to(current_model.device)

        # regular accuracy
        with torch.no_grad():
            output = current_model.generate(**inputs, max_new_tokens=512, temperature=0.0, do_sample=False)
        response = current_tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        if extract_prediction(response) == gt_answer:
            accuracy_correct += 1

        # pass @k 
        answers_k = _pass_at_k_batch(current_model, current_tokenizer, inputs, k)
        pass_at_4_hits += int(any(a == gt_answer for a in answers_k[:4]))
        pass_at_8_hits += int(any(a == gt_answer for a in answers_k))
        counter = collections.Counter(a for a in answers_k if a is not None)
        majority_ans = counter.most_common(1)[0][0] if counter else None
        majority_hits += int(majority_ans == gt_answer)
        total += 1

    current_model.train()

    metrics = {
        "label":         label,
        "accuracy":      accuracy_correct / total,
        "pass_at_4":     pass_at_4_hits / total,
        "pass_at_8":     pass_at_8_hits / total,
        "majority_vote": majority_hits / total,
        # not measure for base/SFT but kept for consistent format
        "cot":           None,   
        "self_correct":  None,   
    }
    print(f"  Accuracy: {metrics['accuracy']*100:.2f}% | "
          f"Pass@4: {metrics['pass_at_4']*100:.2f}% | "
          f"Pass@8: {metrics['pass_at_8']*100:.2f}% | "
          f"Majority: {metrics['majority_vote']*100:.2f}%")
    return metrics

In [ ]:
def analyze_grpo(current_model, current_tokenizer, dataset, label, k=8):
    print(f"\n{'#'*60}\n  GRPO ANALYSIS: {label}\n{'#'*60}")
    current_model.eval()

    total = accuracy_correct = pass_at_4_hits = pass_at_8_hits = majority_hits = 0
    cot_results = {"correct": [], "incorrect": []}
    sc_records = []

    for example in tqdm(dataset, desc=f"Evaluating ({label})"):
        question = example["question"]
        gt_answer = extract_ground_truth(example["answer"])

        messages = [{"role": "system", "content": SYSTEM_THINK}, {"role": "user", "content": question}]
        text = current_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = current_tokenizer([text], return_tensors="pt").to(current_model.device)

        # regular accuracy
        with torch.no_grad():
            output = current_model.generate(**inputs, max_new_tokens=512, temperature=0.0, do_sample=False)
        response = current_tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        is_correct  = (extract_prediction(response) == gt_answer)
        if is_correct:
            accuracy_correct += 1

        # Check self-correction words
        think_text = extract_think_block(response)
        think_tokens = len(current_tokenizer.encode(think_text)) if think_text else 0
        cot_results["correct" if is_correct else "incorrect"].append(think_tokens)
        has_correction = bool(BACKTRACK_PATTERN.search(think_text)) if think_text else False
        sc_records.append((has_correction, is_correct))

        # pass @ k
        answers_k  = _pass_at_k_batch(current_model, current_tokenizer, inputs, k)
        pass_at_4_hits += int(any(a == gt_answer for a in answers_k[:4]))
        pass_at_8_hits += int(any(a == gt_answer for a in answers_k))
        counter = collections.Counter(a for a in answers_k if a is not None)
        majority_ans = counter.most_common(1)[0][0] if counter else None
        majority_hits += int(majority_ans == gt_answer)
        total += 1

    current_model.train()

    def _stats(vals):
        if not vals:
            return {"n": 0, "mean": 0.0, "median": 0, "max": 0}
        s = sorted(vals)
        return {"n": len(s), "mean": sum(s)/len(s), "median": s[len(s)//2], "max": s[-1]}

    n_with_sc = sum(1 for hc, _ in sc_records if hc)
    n_without  = total - n_with_sc

    metrics = {
        "label":         label,
        "accuracy":      accuracy_correct / total,
        "pass_at_4":     pass_at_4_hits   / total,
        "pass_at_8":     pass_at_8_hits   / total,
        "majority_vote": majority_hits    / total,
        "cot": {
            "correct":   _stats(cot_results["correct"]),
            "incorrect": _stats(cot_results["incorrect"]),
        },
        "self_correct": {
            "rate_correcting": n_with_sc / total,
            "acc_with_sc":    sum(ic for hc, ic in sc_records if hc)     / n_with_sc if n_with_sc else float("nan"),
            "acc_without_sc": sum(ic for hc, ic in sc_records if not hc) / n_without if n_without else float("nan"),
        },
    }
    print(f"  Accuracy: {metrics['accuracy']*100:.2f}% | "
          f"Pass@4: {metrics['pass_at_4']*100:.2f}% | "
          f"Pass@8: {metrics['pass_at_8']*100:.2f}% | "
          f"Majority: {metrics['majority_vote']*100:.2f}%")
    print(f"  Avg Think Tokens (correct): {metrics['cot']['correct']['mean']:.1f} | "
          f"Self-Correction Rate: {metrics['self_correct']['rate_correcting']*100:.1f}%")
    return metrics

# Baseline Eval

In [ ]:
print("\n" + "="*40)
print("--- BASELINE SAMPLE ---")
print(f"Question: {sample_question}")
print(f"Answer:\n{test_model_inference(model, tokenizer, sample_question)}")
print("="*40 + "\n")

In [ ]:
baseline_results = analyze_standard(model, tokenizer, test_dataset, label="BASELINE", k=8)
print(f"\n{baseline_results}")

# SFT Training and Eval

In [ ]:
print("Starting Supervised Fine-Tuning...")
trainer.train()

In [ ]:
think_prompt = "Solve the math problem. Put reasoning inside <think> and </think>, then give the final answer after ####."
print("\n" + "="*40)
print("--- POST-SFT EVALUTATION ---")
print(f"Question: {sample_question}")
print(f"Answer:\n{test_model_inference(model, tokenizer, sample_question, system_content=think_prompt)}")
print("="*40 + "\n")

In [ ]:
sft_results = analyze_standard(model, tokenizer, test_dataset, label="POST-SFT", k=8, system_prompt=SYSTEM_THINK)
print(f"\n{sft_results}")

In [ ]:
trainer.save_model(drive_save_path_sft)
print("Saved successfully!")

# GRPO Training and Eval

In [ ]:
print("Starting GRPO Fine-Tuning...")
grpo_trainer.train()

In [ ]:
think_prompt = "Solve the math problem. Put reasoning inside <think> and </think>, then give the final answer after ####."
print("\n" + "="*40)
print("--- POST-GRPO SAMPLE ---")
print(f"Question: {sample_question}")
print(f"Answer:\n{test_model_inference(model, tokenizer, sample_question, system_content=think_prompt)}")
print("="*40 + "\n")

In [ ]:
grpo_results = analyze_grpo(model, tokenizer, test_dataset, label="POST-GRPO", k=8)
print(f"\n{grpo_results}")

In [ ]:
grpo_trainer.save_model(drive_save_path_grpo)
print("Saved successfully!")